# RNN Best Translator

Loads the saved `RNN Outputs/rnn_best.pt` checkpoint and runs English -> Igbo translation with the trained LSTM sequence-to-sequence RNN. This mirrors the testing workflow from `Transformer_Best_Translator.ipynb`: load the best saved model, use a manual translation cell, or use the optional widget translator.


In [1]:
from __future__ import annotations

import importlib.util
import re
import sys
from html import escape
from pathlib import Path
from urllib.parse import quote

if importlib.util.find_spec("torch") is None:
    print("PyTorch is not installed in this notebook kernel.")
    print()
    print("Run this in a new cell, then restart the kernel:")
    print("%pip install --upgrade pip")
    print("%pip install torch ipywidgets")
    raise SystemExit("Install PyTorch, restart the kernel, then rerun the notebook from the top.")

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from IPython.display import HTML, display

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

if importlib.util.find_spec("ipywidgets") is not None:
    import ipywidgets as widgets
else:
    widgets = None
    print("ipywidgets is not installed. The manual translation cell will still work.")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Detected GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.10.0+cu130
CUDA available: True
Detected GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## 1. Find The Saved Best Checkpoint


In [2]:
PROJECT_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path(r"C:\Users\Mr. Paul\Downloads\CS156 Final Assignment"),
]
OUTPUT_DIR_NAMES = ["RNN Outputs"]


def resolve_project_dir(candidates):
    for candidate in candidates:
        for output_dir_name in OUTPUT_DIR_NAMES:
            if (candidate / output_dir_name).exists():
                return candidate
    return None


def resolve_output_dir(project_dir):
    for output_dir_name in OUTPUT_DIR_NAMES:
        output_dir = project_dir / output_dir_name
        if output_dir.exists():
            return output_dir
    raise FileNotFoundError("Could not locate the RNN output folder.")


PROJECT_DIR = resolve_project_dir(PROJECT_DIR_CANDIDATES)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project directory. Run this notebook from the project root or one level below it."
    )

OUTPUT_DIR = resolve_output_dir(PROJECT_DIR)
BEST_CHECKPOINT_PATH = OUTPUT_DIR / "rnn_best.pt"
LATEST_CHECKPOINT_PATH = OUTPUT_DIR / "rnn_latest.pt"

if not BEST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No best RNN checkpoint was found at {BEST_CHECKPOINT_PATH}.")

print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Best checkpoint path:", BEST_CHECKPOINT_PATH)
if LATEST_CHECKPOINT_PATH.exists():
    print("Latest checkpoint path:", LATEST_CHECKPOINT_PATH)


Project directory: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment
Output directory: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\RNN Outputs
Best checkpoint path: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\RNN Outputs\rnn_best.pt
Latest checkpoint path: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\RNN Outputs\rnn_latest.pt


## 2. Load The Saved Checkpoint Config And Vocabularies


In [3]:
SPECIAL_TOKENS = {
    "pad": "<pad>",
    "sos": "<sos>",
    "eos": "<eos>",
    "unk": "<unk>",
}

# The training notebook disabled cuDNN RNN kernels on this Windows/CUDA setup for stability.
DISABLE_CUDNN_RNN = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.set_float32_matmul_precision("high")
    if DISABLE_CUDNN_RNN:
        torch.backends.cudnn.enabled = False


def load_torch_checkpoint(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


# Keep optimizer/scheduler tensors on CPU; only the model is moved to GPU for inference.
checkpoint = load_torch_checkpoint(BEST_CHECKPOINT_PATH, "cpu")
run_config = checkpoint.get("config", {})

EMBED_DIM = int(run_config.get("embed_dim", 256))
HIDDEN_DIM = int(run_config.get("hidden_dim", 384))
NUM_LAYERS = int(run_config.get("num_layers", 2))
DROPOUT = float(run_config.get("dropout", 0.3))
MAX_SOURCE_TOKENS = int(run_config.get("max_source_tokens", 40))
MAX_DECODING_STEPS = int(run_config.get("max_target_tokens", 40))

source_vocab_itos = checkpoint.get("source_vocab_itos")
target_vocab_itos = checkpoint.get("target_vocab_itos")
if not source_vocab_itos or not target_vocab_itos:
    raise ValueError("The checkpoint does not contain the saved source and target vocabularies.")


class Vocabulary:
    def __init__(self, tokens):
        self.itos = list(tokens)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}

    def encode(self, tokens, add_special_tokens=True):
        ids = [self.stoi.get(token, self.unk_idx) for token in tokens]
        if add_special_tokens:
            ids = [self.sos_idx] + ids + [self.eos_idx]
        return ids

    def decode(self, ids):
        tokens = []
        special_values = set(SPECIAL_TOKENS.values())
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                continue
            token = self.itos[idx]
            if token == SPECIAL_TOKENS["eos"]:
                break
            if token not in special_values:
                tokens.append(token)
        return " ".join(tokens)

    @property
    def pad_idx(self):
        return self.stoi[SPECIAL_TOKENS["pad"]]

    @property
    def sos_idx(self):
        return self.stoi[SPECIAL_TOKENS["sos"]]

    @property
    def eos_idx(self):
        return self.stoi[SPECIAL_TOKENS["eos"]]

    @property
    def unk_idx(self):
        return self.stoi[SPECIAL_TOKENS["unk"]]

    def __len__(self):
        return len(self.itos)


source_vocab = Vocabulary(source_vocab_itos)
target_vocab = Vocabulary(target_vocab_itos)

summary = {
    "device": str(device),
    "checkpoint_epoch": checkpoint.get("epoch"),
    "best_validation_loss": checkpoint.get("best_val_loss"),
    "source_vocab_size": len(source_vocab),
    "target_vocab_size": len(target_vocab),
    "max_source_tokens": MAX_SOURCE_TOKENS,
    "max_decoding_steps": MAX_DECODING_STEPS,
    "embed_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
}

display(summary)

history = checkpoint.get("history", [])
if history:
    if pd is not None:
        display(pd.DataFrame(history))
    else:
        display(history)


{'device': 'cuda',
 'checkpoint_epoch': 8,
 'best_validation_loss': 4.898693353116156,
 'source_vocab_size': 34985,
 'target_vocab_size': 32107,
 'max_source_tokens': 40,
 'max_decoding_steps': 40,
 'embed_dim': 256,
 'hidden_dim': 384,
 'num_layers': 2,
 'dropout': 0.3}

,epoch,train_loss,val_loss,learning_rate
0,1,4.813040,5.306933,0.0005
1,2,4.099538,5.193684,0.0005
2,3,3.830722,5.015353,0.0005
3,4,3.652454,5.038316,0.0005
4,5,3.541830,5.034937,0.0005
5,6,3.417814,4.952639,0.0005
6,7,3.318172,4.955296,0.0005
7,8,3.242334,4.898693,0.0005


## 3. RNN Classes Needed To Reload The Saved `.pt` Model


In [4]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(input_dim, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
            batch_first=True,
        )
        self.hidden_bridge = nn.Linear(hidden_dim * 2, hidden_dim)
        self.cell_bridge = nn.Linear(hidden_dim * 2, hidden_dim)

    def _combine_directions(self, state):
        combined_states = []
        for layer in range(self.num_layers):
            forward_state = state[2 * layer]
            backward_state = state[2 * layer + 1]
            combined_states.append(torch.cat([forward_state, backward_state], dim=1))
        return torch.stack(combined_states, dim=0)

    def forward(self, source, source_lengths):
        embedded = self.dropout(self.embedding(source))
        packed = pack_padded_sequence(embedded, source_lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hidden, cell) = self.lstm(packed)
        hidden = torch.tanh(self.hidden_bridge(self._combine_directions(hidden)))
        cell = torch.tanh(self.cell_bridge(self._combine_directions(cell)))
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden, cell):
        input_token = input_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_token))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, source_lengths, target, teacher_forcing_ratio=0.5):
        batch_size = source.size(0)
        target_length = target.size(1)
        target_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(batch_size, target_length, target_vocab_size, device=self.device)
        hidden, cell = self.encoder(source, source_lengths)
        input_token = target[:, 0]

        for step in range(1, target_length):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, step] = output
            teacher_force = False
            top_prediction = output.argmax(dim=1)
            input_token = target[:, step] if teacher_force else top_prediction

        return outputs

    def greedy_decode(self, source, source_lengths, sos_idx, eos_idx, max_steps):
        self.eval()
        with torch.no_grad():
            hidden, cell = self.encoder(source, source_lengths)
            input_token = torch.tensor([sos_idx], dtype=torch.long, device=self.device)
            generated_ids = []

            for _ in range(max_steps):
                output, hidden, cell = self.decoder(input_token, hidden, cell)
                next_token = output.argmax(dim=1)
                token_id = next_token.item()
                if token_id == eos_idx:
                    break
                generated_ids.append(token_id)
                input_token = next_token

        return generated_ids


## 4. Load The Best Saved RNN Model


In [5]:
encoder = Encoder(
    input_dim=len(source_vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=source_vocab.pad_idx,
)

decoder = Decoder(
    output_dim=len(target_vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=target_vocab.pad_idx,
)

translator_model = Seq2Seq(encoder, decoder, device).to(device)
translator_model.load_state_dict(checkpoint["model_state_dict"])
translator_model.eval()

parameter_count = sum(parameter.numel() for parameter in translator_model.parameters())
print("Loaded RNN checkpoint:", BEST_CHECKPOINT_PATH)
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Trainable parameters:", f"{parameter_count:,}")
print("Device:", device)


Loaded RNN checkpoint: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\RNN Outputs\rnn_best.pt
Checkpoint epoch: 8
Trainable parameters: 37,813,483
Device: cuda


## 5. Translation Helpers


In [6]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def normalize_text(text):
    text = str(text).lower().strip()
    text = text.replace("\u2019", "'")
    text = text.replace("\u2018", "'")
    text = text.replace("\u201c", '"')
    text = text.replace("\u201d", '"')
    text = text.replace("\u02bc", "'")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text):
    return TOKEN_PATTERN.findall(normalize_text(text))


def translate_text(text, max_steps=MAX_DECODING_STEPS):
    text = str(text).strip()
    if not text:
        raise ValueError("Enter an English sentence first.")

    tokens = tokenize(text)[:MAX_SOURCE_TOKENS]
    if not tokens:
        raise ValueError("Enter an English sentence first.")

    source_ids = source_vocab.encode(tokens)
    source_tensor = torch.tensor(source_ids, dtype=torch.long, device=device).unsqueeze(0)
    source_length = torch.tensor([len(source_ids)], dtype=torch.long)

    generated_ids = translator_model.greedy_decode(
        source_tensor,
        source_length,
        target_vocab.sos_idx,
        target_vocab.eos_idx,
        max_steps,
    )
    return target_vocab.decode(generated_ids)


def google_translate_back_link(igbo_text):
    return f"https://translate.google.com/?sl=ig&tl=en&text={quote(igbo_text)}&op=translate"


def build_translation_html(english_text, igbo_text):
    back_link = google_translate_back_link(igbo_text)
    safe_english = escape(english_text)
    safe_igbo = escape(igbo_text if igbo_text else "[empty output]")
    return f"""
    <div style='font-family: sans-serif; line-height: 1.5;'>
      <p><strong>English input</strong></p>
      <div style='padding: 10px; border: 1px solid #ddd; border-radius: 6px; margin-bottom: 12px;'>{safe_english}</div>
      <p><strong>RNN Igbo output</strong></p>
      <div style='padding: 10px; border: 1px solid #ddd; border-radius: 6px; margin-bottom: 12px;'>{safe_igbo}</div>
      <a href='{back_link}' target='_blank'>Open Google Translate back-translation (Igbo -> English)</a>
    </div>
    """


def show_translation_result(english_text):
    english_text = str(english_text).strip()
    if not english_text:
        raise ValueError("Enter an English sentence first.")

    igbo_text = translate_text(english_text)
    back_link = google_translate_back_link(igbo_text)
    display(HTML(build_translation_html(english_text, igbo_text)))

    return {
        "english_input": english_text,
        "igbo_output": igbo_text,
        "back_translation_url": back_link,
    }


## 6. Manual Translator Cell


In [7]:
english_text = "How are you?"

if english_text == "type your English sentence here":
    print("Replace `english_text` with a real sentence, then rerun this cell.")
else:
    result = show_translation_result(english_text)
    display(result)


{'english_input': 'How are you?',
 'igbo_output': 'olee otú ị ga - esi mee nke a ?',
 'back_translation_url': 'https://translate.google.com/?sl=ig&tl=en&text=olee%20ot%C3%BA%20%E1%BB%8B%20ga%20-%20esi%20mee%20nke%20a%20%3F&op=translate'}

## 7. Optional Widget Tool


In [8]:
if widgets is None:
    print("ipywidgets is not installed in this environment. Use the manual translator cell above instead.")
else:
    heading = widgets.HTML("<h3 style='margin: 0 0 8px 0;'>RNN Translator</h3>")
    input_label = widgets.HTML("<b>English input</b>")
    input_box = widgets.Textarea(
        value="",
        placeholder="Type English text here...",
        layout=widgets.Layout(width="100%", height="120px"),
    )
    translate_button = widgets.Button(
        description="Translate",
        button_style="info",
        icon="language",
        tooltip="Run the saved RNN model on the input text",
    )
    status_box = widgets.HTML("")
    result_box = widgets.HTML("<i>Enter text above and click Translate.</i>")

    def on_translate_click(_):
        english_text = input_box.value.strip()
        if not english_text:
            status_box.value = "<span style='color: #b00020;'>Enter an English sentence first.</span>"
            result_box.value = ""
            return

        translate_button.disabled = True
        status_box.value = "<span style='color: #555;'>Translating...</span>"

        try:
            igbo_text = translate_text(english_text)
            result_box.value = build_translation_html(english_text, igbo_text)
            status_box.value = ""
        except Exception as exc:
            status_box.value = f"<span style='color: #b00020;'>Translation failed: {escape(str(exc))}</span>"
            result_box.value = ""
        finally:
            translate_button.disabled = False

    translate_button.on_click(on_translate_click)
    display(widgets.VBox([heading, input_label, input_box, translate_button, status_box, result_box]))


## 8. Quick Sample Outputs


In [9]:
sample_sentences = [
    "good morning",
    "god is good",
    "where are you going",
    "the lord is my shepherd",
]

for sentence in sample_sentences:
    print("English:", sentence)
    print("RNN Igbo:", translate_text(sentence))
    print("-" * 60)


English: good morning
RNN Igbo: ìhè ụtụtụ
------------------------------------------------------------
English: god is good
RNN Igbo: chineke dị mma
------------------------------------------------------------
English: where are you going
RNN Igbo: ebe ị na - aga
------------------------------------------------------------
English: the lord is my shepherd
RNN Igbo: onyenwe anyị bụ onye ọzụzụ atụrụ
------------------------------------------------------------
